In [7]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

In [2]:
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

# **RAG IMPLEMENTATION WITH PDF DATA**

### STEP 1: Preparing document for text

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/Azure_essentials.pdf"

loader = PyPDFLoader(pdf_path)
#Create a document
docs = loader.load()

### STEP 2: Splitting documents into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=100,
)

chunks = splitter.split_documents(docs)

In [5]:
len(chunks)

546

## Creating own Metadata for PDF chunks

In [6]:
#PyPDF creates metadata for us
chunks[0].metadata

{'producer': 'Microsoft® Word 2016',
 'creator': 'Microsoft® Word 2016',
 'creationdate': '2016-08-10T16:18:28-04:00',
 'author': 'Michael Collier and Robin Shahan',
 'moddate': '2016-08-10T18:06:36-04:00',
 'title': 'Microsoft_Azure_Essentials_Fundamentals_of_Azure_Second_Edition_e-optimized_mobile_PDF',
 'source': './Docs/Azure_essentials.pdf',
 'total_pages': 546,
 'page': 0,
 'page_label': 'Cover'}

In [7]:
for i in docs:
    i.metadata = {
        "source" : "azure-essentials.pdf",
        "developer" : "Microsoft"
    }

#Ran a loop in docs beacuse the list of docs is smaller than chunks

### STEP 3: Creating Embeddings for the chuncks

In [3]:
from langchain_openai import OpenAIEmbeddings

#the default is text-embedding-3-small
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

### STEP 4: Creating and Store Embedding in **LOCAL** Vector Store

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./Vector/"
)

## Reuse the vector DB

In [4]:
from langchain_community.vectorstores import Chroma
vectorstore_persist = Chroma(
    persist_directory="./Vector/",
    embedding_function=embedding_model
)

/var/folders/j9/fgc6h41s71n0gqpzy_jtd5hw0000gn/T/ipykernel_15864/3878519306.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist = Chroma(


In [5]:
vectorstore_persist.similarity_search("Why do we need Azure? Tell me in one line", k=3)

[Document(metadata={'source': './Docs/Azure_essentials.pdf', 'author': 'Michael Collier and Robin Shahan', 'producer': 'Microsoft® Word 2016', 'page': 35, 'title': 'Microsoft_Azure_Essentials_Fundamentals_of_Azure_Second_Edition_e-optimized_mobile_PDF', 'moddate': '2016-08-10T18:06:36-04:00', 'creationdate': '2016-08-10T16:18:28-04:00', 'creator': 'Microsoft® Word 2016', 'page_label': '6', 'total_pages': 546}, page_content='6 of 540 C H A P T E R 1  |  Getting started with Microsoft Azure \n \nanother eight regions. Only the largest global \nenterprises are able to deploy datacenters in this \nmanner, so using Azure makes it easy for \nenterprises of any size to deploy their services \nclose to their customers, wherever they are in the \nworld. And you can do that without ever leaving \nyour office. \nFor startups, Azure allows you to start with very \nlow cost and scale rapidly as you gain customers. \nYou would not face a large up-front capital \ninvestment to create a new VM—or even